# Parallax 3D GIF Generator

Turn any single image into a 3D parallax GIF using monocular depth estimation, subject segmentation, and LaMa inpainting.

**Pipeline:**
1. Depth estimation (Depth Anything V2)
2. Subject segmentation (RMBG-2.0) with soft alpha preservation
3. Multi-layer depth inpainting (LaMa)
4. Sub-pixel orbital rendering (`cv2.remap`)
5. GIF assembly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanpd13/parallax-3d-gif/blob/main/parallax_3d_gif.ipynb)

## 1. Setup

Install dependencies and upload your image.

In [ ]:
import subprocess, sys

def _is_installed(pkg):
    try:
        __import__(pkg)
        return True
    except ImportError:
        return False

if not _is_installed('kornia') or not _is_installed('simple_lama_inpainting'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'torchvision', 'opencv-python-headless', 'Pillow', 'scipy',
        'tqdm', 'huggingface_hub', 'transformers>=4.45,<5.0',
        'simple-lama-inpainting', 'kornia'])
else:
    print('Dependencies already installed')

In [ ]:
import os

# HF login for gated models (RMBG-2.0 requires it)
# Pass token via: HF_TOKEN env var, Colab secrets, or interactive login
_hf_token = os.environ.get('HF_TOKEN')

if not _hf_token:
    try:
        from google.colab import userdata
        _hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print('Logged in to HF with token')
elif 'COLAB_RELEASE_TAG' in os.environ:
    from huggingface_hub import login
    login()  # interactive prompt
else:
    # Try to use cached credentials
    from huggingface_hub import HfFolder
    if HfFolder.get_token():
        print('Using cached HF credentials')
    else:
        print('WARNING: No HF token found. Gated models (RMBG-2.0) will fail.')

In [ ]:
import glob

# If INPUT is already defined (e.g. by papermill or a preceding cell), use it.
# Otherwise, look for any image in the current directory, or prompt Colab upload.
if 'INPUT' not in dir() or not os.path.isfile(INPUT):
    # Check for existing images in cwd
    existing = glob.glob('*.jpg') + glob.glob('*.jpeg') + glob.glob('*.png') + glob.glob('*.webp')
    # Exclude pipeline outputs
    existing = [f for f in existing if f not in ('depth.png', 'mask.png', 'bg_inpainted.png')]

    if existing:
        INPUT = existing[0]
        print(f'Found image: {INPUT}')
    elif 'COLAB_RELEASE_TAG' in os.environ:
        from google.colab import files
        uploaded = files.upload()
        INPUT = list(uploaded.keys())[0]
    else:
        raise FileNotFoundError('No input image found. Place a .jpg/.png in the working directory or set INPUT.')

print(f'Ready: {INPUT}')

## 2. Configuration

Adjust these parameters to control the 3D effect.

In [ ]:
from pathlib import Path

# -- Animation --
FRAMES        = 9        # number of viewpoints to render
ARC_DEG       = 7.0      # total camera arc in degrees
FPS           = 12       # GIF frame rate
PING_PONG     = True     # bounce animation back and forth

# -- Image --
MAX_DIM       = 2048     # max image dimension (longer side)

# -- Parallax --
BG_PARALLAX   = 2.5      # background shift multiplier vs foreground
BG_BLUR       = 3        # simulated depth-of-field blur on background
NUM_LAYERS    = 4        # depth layers (more = smoother parallax, slower inpainting)

# -- Compositing --
AO_WIDTH      = 6        # ambient occlusion shadow width
AO_STRENGTH   = 0.35     # ambient occlusion shadow intensity

# -- Inpainting --
MASK_DILATE_K = 21       # dilation kernel size for inpaint mask
MASK_DILATE_I = 5        # dilation iterations

OUTPUT = Path(INPUT).stem + '_3d.gif'

## 3. Run Pipeline

In [ ]:
import cv2
import torch
import numpy as np
import time
import os
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from tqdm import tqdm
from IPython.display import display, Image as IPImage

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# ── Helper functions ──

def guided_filter(guide, target, radius=8, eps=0.01):
    """Edge-preserving smoothing using a guided filter."""
    guide = guide.astype(np.float64)
    target = target.astype(np.float64)
    ksize = 2 * radius + 1
    mean_g = cv2.boxFilter(guide, -1, (ksize, ksize))
    mean_t = cv2.boxFilter(target, -1, (ksize, ksize))
    mean_gt = cv2.boxFilter(guide * target, -1, (ksize, ksize))
    mean_gg = cv2.boxFilter(guide * guide, -1, (ksize, ksize))
    a = (mean_gt - mean_g * mean_t) / (mean_gg - mean_g * mean_g + eps)
    b = mean_t - a * mean_g
    mean_a = cv2.boxFilter(a, -1, (ksize, ksize))
    mean_b = cv2.boxFilter(b, -1, (ksize, ksize))
    return (mean_a * guide + mean_b).astype(np.float32)


def iterative_inpaint(img_bgr, m, passes=5):
    """Multi-pass OpenCV inpainting for large masked areas."""
    result = img_bgr.copy()
    remaining = m.copy()
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    for _ in range(passes):
        eroded = cv2.erode(remaining, k, iterations=3)
        band = np.clip(remaining.astype(np.float32) - eroded.astype(np.float32), 0, 1).astype(np.uint8)
        if band.sum() < 10:
            break
        result = cv2.inpaint(result, band, 7, cv2.INPAINT_NS)
        remaining = eroded
    if remaining.sum() > 0:
        result = cv2.inpaint(result, remaining, 15, cv2.INPAINT_TELEA)
    return result


def inpaint_lama(img_rgb, mask_binary):
    """LaMa inpainting with OpenCV fallback."""
    try:
        from simple_lama_inpainting import SimpleLama
        print('  Using LaMa inpainting...')
        lama = SimpleLama()
        img_pil = Image.fromarray(img_rgb)
        mask_pil = Image.fromarray((mask_binary * 255).astype(np.uint8))
        result_pil = lama(img_pil, mask_pil)
        return np.array(result_pil)
    except Exception as e:
        print(f'  LaMa failed ({e}), falling back to OpenCV...')
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        result_bgr = iterative_inpaint(img_bgr, mask_binary.astype(np.uint8))
        return cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)


def build_ao_shadow(mask, width=6, strength=0.35):
    """Generate ambient occlusion shadow at mask edges."""
    dist = cv2.distanceTransform((mask > 0.5).astype(np.uint8), cv2.DIST_L2, 5)
    dist = np.clip(dist / max(width, 1), 0, 1)
    ao = 1.0 - (1.0 - dist) * strength * (mask > 0.5).astype(np.float32)
    return cv2.GaussianBlur(np.clip(ao, 0, 1).astype(np.float32), (5, 5), 1.5)

In [ ]:
# ── Load image ──

raw = Image.open(INPUT).convert('RGB')
w, h = raw.size
if max(w, h) > MAX_DIM:
    s = MAX_DIM / max(w, h)
    w, h = int(w * s) & ~1, int(h * s) & ~1
    raw = raw.resize((w, h), Image.LANCZOS)
img = np.array(raw)
H, W = img.shape[:2]
print(f'Image: {W}x{H}')

In [ ]:
# ── Stage 1: Depth Estimation ──

t0 = time.time()
print('Stage 1: Depth Estimation')

from transformers import AutoImageProcessor, AutoModelForDepthEstimation

depth_proc = AutoImageProcessor.from_pretrained('depth-anything/Depth-Anything-V2-Small-hf')
depth_model = AutoModelForDepthEstimation.from_pretrained(
    'depth-anything/Depth-Anything-V2-Small-hf'
).to(device).eval()

inputs = depth_proc(images=Image.fromarray(img), return_tensors='pt').to(device)
with torch.no_grad():
    pred = depth_model(**inputs).predicted_depth

depth_raw = torch.nn.functional.interpolate(
    pred.unsqueeze(1), size=(H, W), mode='bicubic'
)[0, 0].cpu().numpy().astype(np.float32)
depth = (depth_raw - depth_raw.min()) / (depth_raw.max() - depth_raw.min() + 1e-8)

del depth_model, depth_proc
torch.cuda.empty_cache()

# Auto-detect polarity (near=0, far=1)
center_d = depth[H//3:2*H//3, W//3:2*W//3].mean()
edge_d = np.concatenate([depth[0,:], depth[-1,:], depth[:,0], depth[:,-1]]).mean()
if center_d > edge_d:
    depth = 1.0 - depth
    print('  (inverted depth polarity)')

# Refine depth edges
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
depth = guided_filter(gray, depth, radius=8, eps=0.02)
depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

print(f'  Done in {time.time()-t0:.1f}s')
cv2.imwrite('depth.png', cv2.applyColorMap((depth * 255).astype(np.uint8), cv2.COLORMAP_INFERNO))

In [ ]:
# ── Stage 2: Subject Segmentation ──

t0 = time.time()
print('Stage 2: Segmentation')

from transformers import AutoModelForImageSegmentation
from torchvision.transforms.functional import normalize

rmbg = AutoModelForImageSegmentation.from_pretrained(
    'briaai/RMBG-2.0', trust_remote_code=True
).to(device).eval()

t_img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
t_img = normalize(t_img, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
t_img = torch.nn.functional.interpolate(
    t_img.unsqueeze(0), (1024, 1024), mode='bilinear'
).to(device)

with torch.no_grad():
    mask_raw = torch.sigmoid(rmbg(t_img)[-1][0, 0]).cpu().numpy()

# Soft alpha — preserve the model's continuous 0-to-1 edge detail
mask_soft = cv2.resize(mask_raw, (W, H)).astype(np.float32)
kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mask_soft = cv2.morphologyEx(mask_soft, cv2.MORPH_CLOSE, kern, iterations=3)
# NO thresholding — keep the gradient at edges

# Binary mask for inpainting and layer assignment
mask_binary = (mask_soft > 0.5).astype(np.uint8)
mask_binary = cv2.morphologyEx(mask_binary, cv2.MORPH_OPEN, kern, iterations=1)

del rmbg
torch.cuda.empty_cache()

print(f'  Foreground: {mask_binary.mean()*100:.1f}%  Done in {time.time()-t0:.1f}s')
cv2.imwrite('mask.png', (mask_soft * 255).astype(np.uint8))

In [ ]:
# ── Stage 3: Multi-Layer Inpainting ──

t0 = time.time()
print('Stage 3: Multi-Layer Inpainting')

# --- Quantize depth into layers using percentile thresholds ---
# Exclude subject pixels from threshold computation so the subject
# doesn't skew the depth distribution of background layers.
bg_pixels = depth[mask_binary == 0]
if len(bg_pixels) > 0:
    pcts = np.linspace(0, 100, NUM_LAYERS + 1)[1:-1]  # e.g. [25, 50, 75] for 4 layers
    thresholds = np.percentile(bg_pixels, pcts)
else:
    thresholds = np.linspace(0, 1, NUM_LAYERS + 1)[1:-1]

print(f'  Depth thresholds: {thresholds}')

# Build per-layer binary masks (layer 0 = nearest, layer N-1 = farthest)
layer_masks = []
for i in range(NUM_LAYERS):
    if i == 0:
        m = depth <= thresholds[0]
    elif i == NUM_LAYERS - 1:
        m = depth > thresholds[-1]
    else:
        m = (depth > thresholds[i - 1]) & (depth <= thresholds[i])
    layer_masks.append(m.astype(np.float32))

# Force subject into the nearest layer (layer 0)
for i in range(1, NUM_LAYERS):
    layer_masks[i][mask_binary > 0] = 0.0
layer_masks[0][mask_binary > 0] = 1.0

# Clean layer masks: close small holes, remove tiny isolated regions
kern_clean = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
for i in range(NUM_LAYERS):
    lm = layer_masks[i].astype(np.uint8)
    lm = cv2.morphologyEx(lm, cv2.MORPH_CLOSE, kern_clean, iterations=2)
    lm = cv2.morphologyEx(lm, cv2.MORPH_OPEN, kern_clean, iterations=1)
    layer_masks[i] = lm.astype(np.float32)

# Merge tiny layers (<2% of pixels) into nearest neighbor
for i in range(NUM_LAYERS):
    if layer_masks[i].mean() < 0.02 and i != 0:  # never merge the subject layer
        merge_target = i - 1 if i > 0 else i + 1
        if merge_target < NUM_LAYERS:
            layer_masks[merge_target] = np.clip(
                layer_masks[merge_target] + layer_masks[i], 0, 1
            )
            layer_masks[i] = np.zeros_like(layer_masks[i])
            print(f'  Layer {i} merged into layer {merge_target} (<2% pixels)')

for i in range(NUM_LAYERS):
    pct = layer_masks[i].mean() * 100
    print(f'  Layer {i}: {pct:.1f}% of image')

# --- Build backing plates back-to-front ---
# Start with farthest layer; for each layer going forward, inpaint
# the hole left by the layer(s) in front of it.
dilate_kern = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (MASK_DILATE_K, MASK_DILATE_K)
)

layers = []  # list of dicts: {image, depth, alpha, disparity_mult}
composite_so_far = img.copy()  # running back-to-front composite for inpainting

for i in reversed(range(NUM_LAYERS)):
    lm_binary = (layer_masks[i] > 0.5).astype(np.uint8)

    if lm_binary.sum() == 0:
        continue  # skip empty layers

    # Determine what needs inpainting: everything NOT in this layer or behind it
    behind_mask = np.zeros((H, W), dtype=np.uint8)
    for j in range(i):
        behind_mask = np.clip(behind_mask + (layer_masks[j] > 0.5).astype(np.uint8), 0, 1)

    # Dilate the "in front" mask for aggressive inpainting
    inpaint_region = cv2.dilate(behind_mask, dilate_kern, iterations=MASK_DILATE_I)

    if inpaint_region.sum() > 0:
        layer_image = inpaint_lama(composite_so_far, inpaint_region)
        if layer_image.shape[:2] != (H, W):
            layer_image = cv2.resize(layer_image, (W, H), interpolation=cv2.INTER_LANCZOS4)
    else:
        layer_image = composite_so_far.copy()

    # Apply depth-of-field blur to non-subject layers
    if i != 0 and BG_BLUR > 0:
        blur_k = BG_BLUR * 2 + 1
        layer_image = cv2.GaussianBlur(layer_image, (blur_k, blur_k), BG_BLUR * 0.8)

    # Inpaint depth for this layer
    d_u8 = (depth * 255).astype(np.uint8)
    if inpaint_region.sum() > 0:
        d_u8_masked = d_u8.copy()
        d_u8_masked[inpaint_region > 0] = 0
        layer_depth = cv2.inpaint(d_u8_masked, inpaint_region, 15, cv2.INPAINT_NS).astype(np.float32) / 255.0
    else:
        layer_depth = depth.copy()

    # Alpha: soft for subject layer, binary for others
    if i == 0:
        layer_alpha = mask_soft.copy()
    else:
        layer_alpha = layer_masks[i].copy()

    # Disparity multiplier: layer 0 (nearest) = 1.0, farthest = BG_PARALLAX
    if NUM_LAYERS > 1:
        disparity_mult = 1.0 + (BG_PARALLAX - 1.0) * (i / (NUM_LAYERS - 1))
    else:
        disparity_mult = 1.0

    layers.append({
        'image': layer_image,
        'depth': layer_depth,
        'alpha': layer_alpha,
        'disparity_mult': disparity_mult,
        'index': i,
    })

# Reverse so layers are ordered back-to-front (farthest first) for rendering
layers.sort(key=lambda l: -l['index'])

# Ambient occlusion (applied to subject layer at composite time)
ao_map = build_ao_shadow(mask_binary.astype(np.float32), width=AO_WIDTH, strength=AO_STRENGTH)

print(f'  {len(layers)} active layers  Done in {time.time()-t0:.1f}s')
Image.fromarray(layers[-1]['image'] if layers else img).save('bg_inpainted.png')

In [ ]:
# ── Stage 4: Orbital Rendering (sub-pixel, multi-layer) ──

t0 = time.time()
print('Stage 4: Orbital Rendering')

max_shift = W * np.tan(np.radians(ARC_DEG)) * 0.3
angles = np.linspace(-1.0, 1.0, FRAMES)

# Subject centroid anchor — keeps subject stable in frame
fg_ys, fg_xs = np.where(mask_binary > 0)
if len(fg_xs) > 0:
    subject_center_disparity = (1.0 - np.median(depth[fg_ys, fg_xs])) * max_shift
else:
    subject_center_disparity = 0.0

# Static coordinate grids for cv2.remap
u_grid = np.arange(W, dtype=np.float32)[None, :].repeat(H, axis=0)  # (H, W)
v_grid = np.arange(H, dtype=np.float32)[:, None].repeat(W, axis=1)  # (H, W)

# Precompute per-layer disparity fields
for layer in layers:
    layer['disparity'] = (1.0 - layer['depth']) * max_shift * layer['disparity_mult']

frames = []
for t_val in tqdm(angles, desc='Rendering'):
    anchor_offset = subject_center_disparity * t_val
    result = np.zeros((H, W, 3), dtype=np.float32)

    # Composite back-to-front (farthest layer first)
    for layer in layers:
        # Inverse warp: for each destination pixel, look back at source
        shift_field = layer['disparity'] * t_val - anchor_offset
        map_x = (u_grid - shift_field).astype(np.float32)
        map_y = v_grid.astype(np.float32)

        # Warp image
        warped_img = cv2.remap(
            layer['image'].astype(np.float32), map_x, map_y,
            cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT
        )

        # Warp alpha
        warped_alpha = cv2.remap(
            layer['alpha'].astype(np.float32), map_x, map_y,
            cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT
        )

        # Apply AO shadow to subject layer
        if layer['index'] == 0:
            warped_ao = cv2.remap(
                ao_map, map_x, map_y,
                cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=1.0
            )
            for c in range(3):
                warped_img[:, :, c] *= warped_ao

        # Gentle smoothing on alpha (suppress single-pixel noise only)
        alpha = cv2.GaussianBlur(warped_alpha, (5, 5), 1.0)[:, :, None]
        alpha = np.clip(alpha, 0, 1)

        # Composite
        result = result * (1.0 - alpha) + warped_img * alpha

    frames.append(np.clip(result, 0, 255).astype(np.uint8))

print(f'  {len(frames)} frames  Done in {time.time()-t0:.1f}s')

In [ ]:
# ── Stage 5: GIF Assembly ──

print('Stage 5: GIF Assembly')

if PING_PONG and len(frames) > 2:
    gif_frames = frames + frames[-2:0:-1]
else:
    gif_frames = frames

pil_frames = [Image.fromarray(f) for f in gif_frames]
pil_frames[0].save(
    OUTPUT, save_all=True, append_images=pil_frames[1:],
    duration=int(1000 / FPS), loop=0, optimize=True
)

size_mb = os.path.getsize(OUTPUT) / 1024**2
print(f'{OUTPUT}: {len(gif_frames)} frames, {size_mb:.1f} MB')

# Save individual frames
os.makedirs('frames', exist_ok=True)
for i, f in enumerate(frames):
    Image.fromarray(f).save(f'frames/frame_{i:03d}.png')
print(f'Saved {len(frames)} PNGs to frames/')

display(IPImage(filename=OUTPUT))

## 4. Results & Download

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, path) in zip(axes, [
    ('Input', INPUT), ('Depth', 'depth.png'),
    ('Mask', 'mask.png'), ('BG Inpainted', 'bg_inpainted.png')
]):
    ax.imshow(Image.open(path))
    ax.set_title(title, fontsize=14)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import files
    files.download(OUTPUT)

    # Optional: download frames zip for lenticular interlacing
    # import shutil
    # shutil.make_archive('lenticular_frames', 'zip', 'frames')
    # files.download('lenticular_frames.zip')
else:
    print(f'Output saved to: {OUTPUT}')